In [ ]:
# @markdown Run this cell to check this notebook's version.

current_version = "0.1.1"
notebook_name = "OpticalFlow3D_Profiler"

# First of all check if ipywidgets is installed, if not throw an informative error
try:
    import ipywidgets as widgets
except ImportError:
    raise ImportError("ipywidgets is not installed. Please install it using 'pip install ipywidgets' or 'conda install -c conda-forge ipywidgets'.")

from IPython.display import display

try:
    from google.colab import output
    output.enable_custom_widget_manager()
    
    try:
        import yaml
    except ImportError:
        raise ImportError("pyyaml is not installed. Please install it using 'pip install pyyaml' or 'conda install -c conda-forge pyyaml'.")

    import requests
    import base64

    # Create widgets
    github_repo = widgets.Text(
        value="",
        description="*GitHub Repository URL:",
        placeholder="e.g https://github.com/username/repository",
        layout=widgets.Layout(width="70%"),
        style={'description_width': '150px'},
    )

    github_token = widgets.Password(
        value="",
        description="*Personal Access Token:",
        placeholder="[Disabled] e.g. ghp_XXXXXXXXXXXXXXXXXXXX",
        disabled=True,
        layout=widgets.Layout(width="70%"),
        style={'description_width': '150px'},
    )

    test_button = widgets.Button(
        description="Test Connection",
        button_style='success',
    )

    output_area = widgets.HTML()

    # By default the repository is considered public
    if "repository_is_private" not in globals():
        repository_is_private = False
    elif repository_is_private:
        github_token.disabled = False
        github_token.placeholder = "e.g. ghp_XXXXXXXXXXXXXXXXXXXX"

    def on_test_button_clicked(b):
        global repository_is_private
        
        # Check that the GitHub repository URL is provided
        if not github_repo.value:
            output_area.value = "⚠️ Please provide a GitHub repository URL."
            return
        
        # Get the owner and repo name from the URL
        repo_url = github_repo.value.rstrip('/')
        github_owner, github_repo_name = repo_url.split('/')[-2:]
        
        # Online version checking file path
        version_file_path = "notebooks/notebook_latest_versions.yaml"
        version_url = f"https://api.github.com/repos/{github_owner}/{github_repo_name}/contents/{version_file_path}"

        # Do an initial request to check if the repository is public
        if not repository_is_private:
            version_response = requests.get(version_url)
            if version_response.status_code == 404:
                repository_is_private = True
                github_token.disabled = False
                github_token.placeholder = "e.g. ghp_XXXXXXXXXXXXXXXXXXXX"
                output_area.value = f"⚠️ We have detected that the repository might be private. Please provide a Personal Access Token and click 'Test Connection' again."
                return
        else:
            headers = {"Accept": "application/vnd.github.v3+json"}
            if not github_token.value:
                output_area.value = "⚠️ Personal Access Token is required for private repositories."
                return
            headers["Authorization"] = f"token {github_token.value}"

            version_response = requests.get(version_url, headers=headers)

        # Check the response status
        if version_response.status_code == 200:
            content = version_response.json()['content']
            decoded_content = base64.b64decode(content).decode('utf-8')
            config = yaml.safe_load(decoded_content)
            # notebook_latest_versions.yaml groups the notebooks by folder: {"<folder>": {"<notebook>": "<version>"}}
            latest_version = next((str(group[notebook_name]) for group in (config or {}).values()
                                   if isinstance(group, dict) and notebook_name in group), "")

            output_area.value = (f"<b>Notebook version:</b> `{current_version}`<br>"
                                f"<b>Latest version available:</b> `{latest_version}`<br>")

            if latest_version == "":
                output_area.value += "⚠️ This notebook is not listed in the version file.<br>"
            elif current_version == latest_version:
                output_area.value += "✅ This notebook is up-to-date.<br>"
            else:
                output_area.value += f"⚠️ A new version of this notebook is available."
        else:
            output_area.value += "⚠️ Could not retrieve the version file.<br>"

    test_button.on_click(on_test_button_clicked)
    # Widget layout
    widget_box = widgets.VBox([github_repo, github_token, test_button, output_area])
    display(widget_box)
except ImportError:
    display(widgets.HTML('To check the notebook version, please go to the <a href="../Welcome.ipynb" target="_blank" style="color: #0066cc; text-decoration: underline;">Welcome notebook</a>.'))


# OpticalFlow3D Profiler

Quantify nuclear motion in 3D fluorescence time-lapses with the original
[OpticalFlow3D](https://github.com/aicjanelia/OpticalFlow3D) algorithm (Janelia, BSD-3-Clause),
turn the velocity fields into per-field-of-view motion profiles, and compare experimental conditions.

Work through the five steps in order. Every choice you make is saved in the run folder
(`run_manifest.json`), so a result can always be traced back to its images and settings.

**Read before interpreting results**
- Optical flow measures the motion of *intensity patterns*, not tracked nuclei or single-cell trajectories.
- Speeds are *relative optical-flow speeds*: on synthetic nuclei the 3D algorithm underestimates XY speed
  (×0.5–0.85) and overestimates Z speed, depending on nuclear size (see `docs/validation.md`).
  Compare conditions imaged with the same settings, and check the QC confounders (nuclear size,
  intensity, occupancy, reliability) in step 4.
- Z-dependent measurements are **experimental** with shallow Z-stacks.
- PCA is exploratory. Fields of view of one biological repeat are not independent replicates.

In [ ]:
# @title Setup (run once)
import sys

if "google.colab" in sys.modules:
    # Colab only: the installed app already contains the package.
    !pip install -q "git+https://github.com/guijacquemet/LabConstrictor-OpticalFlow3D"
    from google.colab import output
    output.enable_custom_widget_manager()

%matplotlib inline
from IPython.display import display
from opticalflow3d_profiler.notebook_analysis_ui import build

app = build()
print("Ready.")

## Step 1 — Load images

1. Type the path of one TIFF / OME-TIFF time-lapse (one field of view per file) and click **Inspect header**.
2. Check the **axes** in file order using T, Z, C, Y, X (for example `TZYX` or `TZCYX`). If the header
   does not name them, the field stays empty: you must type them.
3. If the file has several channels, **choose the nuclear channel**.
4. Check the **calibration**. Leave a field empty if you do not know it: results then stay in pixels
   (slices) per frame — no value is ever assumed.
5. Click **Confirm and preview** and browse time points and Z slices.

In [ ]:
# @title Step 1 — Load images
display(app["load"].widget())

## Step 2 — Calculate optical flow

The default parameters are those of the original algorithm (`xyzSig=3`, `tSig=1`, `wSig=4`).
With `tSig=1`, 7 consecutive time points give one output, so a 15-frame movie gives 9 outputs
(time points 3 to 11).

- **Single image:** fill in the experimental labels (condition, repeats…). They are never read from the file name.
- **Several fields of view:** give a metadata CSV with the columns
  `sample_id, position_id, condition, biological_repeat, technical_repeat, batch, input_path`
  (optional `mask_path`). Every file must have the axes, channel and calibration confirmed in step 1.

Click **Estimate resources** first: full-size 1840 × 1840 × 11 stacks need about 11–12 GB of memory
and ≈13 minutes per field of view. Images are never downsampled to save memory. Failed fields of view
are listed, not skipped.

In [ ]:
# @title Step 2 — Calculate optical flow
display(app["flow"].widget())

## Step 3 — Extract motion features

- **Foreground** restricts measurements to nuclei: a binary/label mask in the image's coordinates, or one
  raw-intensity threshold used for every field of view. Without it, background and nuclear density can
  influence the results.
- **Reliability filter** removes vectors where the algorithm has little information. `absolute` uses one
  threshold for every field of view (recommended); `percentile` recomputes it per frame (as in the upstream
  example) and records each value.
- Features are extracted once per run. To try other settings, give a **new run name**: the optical flow is
  reused after its files and settings are verified.

In [ ]:
# @title Step 3 — Extract motion features
display(app["features"].widget())

### Optional — features from an existing OpticalFlow3D result

Use this for a folder written earlier by the original `process_flow` (files `<name>_vx_t0003.tiff`, …).
Its source image cannot be verified; this is recorded with the results. Settings come from steps 1 and 3.

In [ ]:
# @title Optional — external OpticalFlow3D result
display(app["external"].widget(app["features"], app["load"]))

## Step 4 — Compare experiments

List one or more run folders (empty = the current run). Choose feature families for the PCA;
QC and orientation-dependent columns are never PCA inputs, axial features only if you tick them.
The condition comparison averages fields of view within each biological repeat first, reports the
difference and Hedges' g, and runs a test only with at least 3 repeats per condition. Potential
confounders (intensity, occupancy, nuclear size, reliability) are compared alongside.

In [ ]:
# @title Step 4 — Compare experiments
display(app["compare"].widget())

## Step 5 — Export results

Writes PCA scores and loadings, the PCA input matrix and scaling, exclusions, warnings, the condition
comparison and the figures into `<run>/analysis/<analysis name>/`. Feature tables, the feature dictionary,
`fovs.csv` and `run_manifest.json` are already in the run folder.

In [ ]:
# @title Step 5 — Export results
display(app["export"].widget())

---
**Please cite** the OpticalFlow3D paper when you use this app:

> Lee, R. M., Eisenman, L. R., Hobson, C. M., Aaron, J. S. and Chew, T.-L. (2026). OpticalFlow3D – a tool for measuring amorphous motion in three-dimensional fluorescence microscopy images. *J. Cell Sci.* 139(21), jcs264851. https://doi.org/10.1242/jcs.264851

and LabConstrictor: Hidalgo-Cenalmor, I. et al. (2026). Packaging Jupyter notebooks as installable desktop apps using LabConstrictor. arXiv. https://doi.org/10.48550/arXiv.2603.10704